In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re
import sys
import numpy as np
import pandas as pd
import xarray as xr
import statsmodels.api as sm
from natsort import natsorted
from tqdm.notebook import tqdm
from os.path import join as pjoin
import plotly.graph_objects as go
from scipy.stats import zscore, skew
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import plotting_functions as pf
import place_cells as pc

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/btsp_proxy'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#00802d', 'B': '#006c79', 'C': '#004da4', 'D': '#430073'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
sem_color = 'rgba(40, 115, 71, 0.6)'
male_mice = ['mc44', 'mc46', 'mc54', 'mc55', 'mc64', 'mc65']
control_mice = ['mc46', 'mc49', 'mc52', 'mc54', 'mc59', 'mc60', 'mc61', 'mc64']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
velocity_thresh = 10
bin_size = 0.06
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

## Set seed
rs = RandomState(MT19937(SeedSequence(24601)))

### Create a distribution of first trials where stable place cells form for an example mouse.

In [ ]:
## Set mouse information
experiment = 'MultiCon_Imaging7'
mouse = 'mc65'
day_of_int = 16
session = f'{mouse}_{data_type}_{day_of_int}.nc'
cell_type = 'place_cells'
correct_dir = True 
only_running = True

## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced and thresholded activity
sdata = sdata[sdata['minimum_trial_activity_met'], :] ## only use cells that meet the minimum trial threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass

neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)

trial_raster, _ = pc.trial_raster(neural_data, bin_size=bin_size, binarized=True, correct_dir=correct_dir, only_running=only_running) ## trial x spatial bin x neurons
trial_start_per_neuron = pc.place_field_starting_trials(trial_raster)

In [ ]:
## Heatmap of cell activity across trials
plot_bool = False
neuron = 91
fig = pf.custom_graph_template(x_title='Location (rad)', y_title='Trial')
if plot_bool:
    trial_bool = (trial_raster > 0).astype(int)
    fig.add_trace(go.Heatmap(x=np.arange(0, trial_bool.shape[1] * bin_size, bin_size), y=np.arange(0, trial_bool.shape[0]), z=trial_bool[:, :, neuron]))
else:
    fig.add_trace(go.Heatmap(x=np.arange(0, trial_raster.shape[1] * bin_size, bin_size), y=np.arange(0, trial_raster.shape[0]), z=trial_raster[:, :, neuron]))
fig.update_yaxes(autorange='reversed')
fig.show()
# fig.write_image(pjoin(fig_path, f'{mouse}_{day_of_int}_{neuron}_trial_raster_{plot_bool}.png'), width=500, height=500)

### Combine across mice.

In [ ]:
cell_type = 'place_cells'
correct_dir = True 
only_running = True
output_dict = {'mouse': [], 'group': [], 'sex': [], 'session': [], 'day': [], 'uid': [], 'trial_start': [], 'spatial_info': [], 'odd_even': []}

for experiment in os.listdir(dpath):
    if experiment not in experiment_folders:
        pass 
    else:
        exp_path = pjoin(dpath, f'{experiment}/output/{data_of_interest}/')
        behav_path = pjoin(dpath, f'{experiment}/output/behav')
        for mouse in tqdm(os.listdir(exp_path)):
            mpath = pjoin(exp_path, f'{mouse}/{data_type}')
            for index, session in enumerate(natsorted(os.listdir(mpath))):
                if session == 'mc44_S_20.nc':
                    pass 
                else:
                    sdata = xr.open_dataset(pjoin(mpath, session))[data_type]
                    sdata = sdata[sdata['minimum_trial_activity_met'], :] ## remove neurons who didn't meet firing across trials criteria
                    if cell_type == 'place_cells':
                        sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
                    elif cell_type == 'nonplace_cells':
                        sdata = sdata[~sdata['skaggs_place'], :]
                    else:
                        pass

                    neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                                    velocity_thresh=velocity_thresh)

                    trial_raster, _ = pc.trial_raster(neural_data, bin_size=bin_size, binarized=True, correct_dir=correct_dir, only_running=only_running) ## trial x spatial bin x neurons
                    trial_start_per_neuron = pc.place_field_starting_trials(trial_raster)

                    spatial_info = sdata['skaggs_info'].values
                    odd_even = sdata['odd_even'].values

                    for uid in np.arange(trial_start_per_neuron.shape[0]):
                        output_dict['mouse'].append(mouse)
                        output_dict['group'].append(sdata.attrs['group'])
                        output_dict['sex'].append(sdata.attrs['sex'])
                        output_dict['session'].append(sdata.attrs['session_two'])
                        output_dict['day'].append(int(re.search('[0-9]+', session.split('_')[2])[0]))
                        output_dict['uid'].append(uid)
                        output_dict['trial_start'].append(trial_start_per_neuron[uid])
                        output_dict['spatial_info'].append(spatial_info[uid])
                        output_dict['odd_even'].append(odd_even[uid])
trial_start_df = pd.DataFrame(output_dict)
mouse_df = trial_start_df.groupby(['group', 'day', 'mouse'], as_index=False).agg({'trial_start': 'mean'})
avg_trial_start = mouse_df.groupby(['group', 'day'], as_index=False).agg({'trial_start': ['mean', 'sem']})

### Can load trial_start_df from a previous run.

In [ ]:
trial_start_df = pd.read_feather(pjoin(int_data, 'trial_start_df.feat'))

In [ ]:
## Heatmap of trial start on y axis, spatial information on x axis, z value is percentage (number of cells with those values) on day 16
min_trials = None
nbins = 20
day_of_int = 16

sub_df = trial_start_df[trial_start_df['day'] == day_of_int]
if min_trials is not None:
    sub_df = sub_df[sub_df['trial_start'] <= min_trials]
bins_si = np.linspace(np.min(sub_df['spatial_info']), np.max(sub_df['spatial_info']), nbins)
bins_start = np.linspace(np.min(sub_df['trial_start']), np.max(sub_df['trial_start']), nbins)

fig = pf.custom_graph_template(x_title='Spatial Information', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Two-context', 'Multi-context'], width=1000, height=500)
for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = sub_df[sub_df['group'] == group]
    H, _, _, = np.histogram2d(x=gdata['spatial_info'], y=gdata['trial_start'], bins=[bins_si, bins_start])
    Hnorm = H / np.sum(H) * 100 ## in percent
    Hnorm[Hnorm == 0] = np.nan
    fig.add_trace(go.Heatmap(x=bins_si[:-1], y=bins_start[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=1, col=idx + 1) ## have to transpose H, see notes on histogram2d
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
fig.update_yaxes(title='Place Field Trial Start', col=1)
fig.show()
fig.write_image(pjoin(fig_path, f'trial_start_mc_tc_day{day_of_int}_heatmaps_bins{nbins}_spatial_info.png'), width=1000, height=500)

In [ ]:
## Heatmap of trial start on y axis, odd-even trial stability on x axis, z value is percentage (number of cells with those values) on day 16
min_trials = None
nbins = 20
day_of_int = 16

sub_df = trial_start_df[trial_start_df['day'] == day_of_int]
if min_trials is not None:
    sub_df = sub_df[sub_df['trial_start'] <= min_trials]
bins_stab = np.linspace(np.min(sub_df['odd_even']), np.max(sub_df['odd_even']), nbins)
bins_start = np.linspace(np.min(sub_df['trial_start']), np.max(sub_df['trial_start']), nbins)

fig = pf.custom_graph_template(x_title='Odd-Even Trial Stability', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['Two-context', 'Multi-context'], width=1000, height=500)
for idx, group in enumerate(['Two-context', 'Multi-context']):
    gdata = sub_df[sub_df['group'] == group]
    H, _, _, = np.histogram2d(x=gdata['odd_even'], y=gdata['trial_start'], bins=[bins_stab, bins_start])
    Hnorm = H / np.sum(H) * 100 ## in percent
    Hnorm[Hnorm == 0] = np.nan
    fig.add_trace(go.Heatmap(x=bins_stab[:-1], y=bins_start[:-1], z=Hnorm.T, coloraxis='coloraxis1'), row=1, col=idx + 1) ## have to transpose H, see notes on histogram2d
fig.update_layout(coloraxis1=dict(colorscale='viridis'))
fig.update_yaxes(title='Place Field Trial Start', col=1)
fig.show()
fig.write_image(pjoin(fig_path, f'trial_start_mc_tc_day{day_of_int}_heatmaps_bins{nbins}_stability.png'), width=1000, height=500)

In [ ]:
## Create a scatter plot of trial start vs spatial information for a given day
day = 16

fig = pf.custom_graph_template(x_title='Odd-Even Trial Stability', y_title='Trial Start', width=800, height=700)
for mouse in trial_start_df['mouse'].unique():
    mdata = trial_start_df[(trial_start_df['mouse'] == mouse) & (trial_start_df['day'] == day)]
    group = mdata['group'].unique()[0]
    fig.add_trace(go.Scattergl(x=mdata['odd_even'], y=mdata['trial_start'], mode='markers', marker_color=ce_colors_dict[group],
                               name=group, showlegend=False, opacity=0.6, legendgroup=group))
fig['data'][0]['showlegend'] = True
fig['data'][2]['showlegend'] = True
fig.show()